In [4]:
import pandas as pd

In [5]:
import numpy as np

In [12]:
from pathlib import Path

input_path = Path("../data/E-commerce_Delivery_Shipping_Data_2026.csv")
if not input_path.exists():
    raise FileNotFoundError("Place E-commerce_Delivery_Shipping_Data_2026.csv in the project data/ folder, then run this notebook from the python/ folder.")
df = pd.read_csv(input_path)
print(df.head())

     order_id  order_date customer_id customer_segment customer_city  \
0  ORD-000001  15-11-2026  CUS-002353   Small Business         Tokyo   
1  ORD-000002  08-04-2026  CUS-003258         Consumer     Vancouver   
2  ORD-000003  05-01-2026  CUS-001221         Consumer       Houston   
3  ORD-000004  29-05-2026  CUS-001717   Small Business         Delhi   
4  ORD-000005  22-02-2026  CUS-000716   Small Business        Nagoya   

  customer_country warehouse_id warehouse_city product_category  \
0            Japan       WH-004        Toronto   Home & Kitchen   
1           Canada       WH-007          Paris           Beauty   
2    United States       WH-005         London   Home & Kitchen   
3            India       WH-002    Los Angeles          Grocery   
4            Japan       WH-006         Berlin           Beauty   

   product_weight_kg  ...  package_size  payment_method order_priority  \
0               8.23  ...         Large  Digital Wallet         Normal   
1               

In [14]:
# Check the number of rows and columns
print("Dataset Shape:", df.shape)

# Check all column names
print("\nColumn Names:")
print(df.columns.tolist())

# Check the data types
print("\nData Types:")
print(df.dtypes)

Dataset Shape: (50000, 30)

Column Names:
['order_id', 'order_date', 'customer_id', 'customer_segment', 'customer_city', 'customer_country', 'warehouse_id', 'warehouse_city', 'product_category', 'product_weight_kg', 'order_value_usd', 'shipping_method', 'carrier', 'distance_km', 'promised_delivery_days', 'actual_delivery_days', 'delivery_status', 'late_delivery', 'delivery_delay_days', 'shipping_cost_usd', 'package_size', 'payment_method', 'order_priority', 'weather_condition', 'customer_rating', 'return_requested', 'return_reason', 'delivery_attempts', 'warehouse_processing_hours', 'tracking_status']

Data Types:
order_id                       object
order_date                     object
customer_id                    object
customer_segment               object
customer_city                  object
customer_country               object
warehouse_id                   object
warehouse_city                 object
product_category               object
product_weight_kg             float6

In [16]:
df["order_date"] = pd.to_datetime(
    df["order_date"],
    format="%d-%m-%Y"
)

print(df["order_date"].dtype)
print(df["order_date"].head())


datetime64[ns]
0   2026-11-15
1   2026-04-08
2   2026-01-05
3   2026-05-29
4   2026-02-22
Name: order_date, dtype: datetime64[ns]


In [17]:
# Extract the month from the order date
df["order_month"] = df["order_date"].dt.month_name()

# Check the result
print(df[["order_date", "order_month"]].head(10))

  order_date order_month
0 2026-11-15    November
1 2026-04-08       April
2 2026-01-05     January
3 2026-05-29         May
4 2026-02-22    February
5 2026-03-06       March
6 2026-07-10        July
7 2026-12-03    December
8 2026-12-18    December
9 2026-11-05    November


In [19]:
print(df.head())

     order_id order_date customer_id customer_segment customer_city  \
0  ORD-000001 2026-11-15  CUS-002353   Small Business         Tokyo   
1  ORD-000002 2026-04-08  CUS-003258         Consumer     Vancouver   
2  ORD-000003 2026-01-05  CUS-001221         Consumer       Houston   
3  ORD-000004 2026-05-29  CUS-001717   Small Business         Delhi   
4  ORD-000005 2026-02-22  CUS-000716   Small Business        Nagoya   

  customer_country warehouse_id warehouse_city product_category  \
0            Japan       WH-004        Toronto   Home & Kitchen   
1           Canada       WH-007          Paris           Beauty   
2    United States       WH-005         London   Home & Kitchen   
3            India       WH-002    Los Angeles          Grocery   
4            Japan       WH-006         Berlin           Beauty   

   product_weight_kg  ...  payment_method order_priority weather_condition  \
0               8.23  ...  Digital Wallet         Normal              Snow   
1             

In [20]:
# Extract the weekday from the order date
df["order_day_of_week"] = df["order_date"].dt.day_name()

# Check the result
print(df[["order_date", "order_month", "order_day_of_week"]].head(10))

  order_date order_month order_day_of_week
0 2026-11-15    November            Sunday
1 2026-04-08       April         Wednesday
2 2026-01-05     January            Monday
3 2026-05-29         May            Friday
4 2026-02-22    February            Sunday
5 2026-03-06       March            Friday
6 2026-07-10        July            Friday
7 2026-12-03    December          Thursday
8 2026-12-18    December            Friday
9 2026-11-05    November          Thursday


In [22]:
conditions = [
    df["delivery_delay_days"] <= 0,
    df["delivery_delay_days"].between(1, 2),
    df["delivery_delay_days"].between(3, 5),
    df["delivery_delay_days"] > 5
]

categories = [
    "On Time / Early",
    "Minor Delay",
    "Moderate Delay",
    "Severe Delay"
]

df["delay_severity"] = np.select(
    conditions,
    categories,
    default="Check Data"
)

print(df[["delivery_delay_days", "delay_severity"]].head(10))


   delivery_delay_days   delay_severity
0                    5   Moderate Delay
1                    0  On Time / Early
2                    4   Moderate Delay
3                    3   Moderate Delay
4                    4   Moderate Delay
5                    2      Minor Delay
6                    1      Minor Delay
7                    5   Moderate Delay
8                    0  On Time / Early
9                    1      Minor Delay


In [23]:
print(df["delay_severity"].value_counts())

delay_severity
On Time / Early    21838
Minor Delay        19323
Moderate Delay      7201
Severe Delay        1638
Name: count, dtype: int64


In [24]:
print("Zero order values:", (df["order_value_usd"] == 0).sum())

print("Missing order values:", df["order_value_usd"].isna().sum())

Zero order values: 0
Missing order values: 0


In [ ]:
df["shipping_cost_pct_of_order_value"] = (
    df["shipping_cost_usd"] / df["order_value_usd"]
) * 100

print(
    df[["order_value_usd", "shipping_cost_usd", "shipping_cost_pct_of_order_value"]].head(10)
)


In [ ]:
print(df["shipping_cost_pct_of_order_value"].describe())


In [28]:
print(df["distance_km"].describe())

count    50000.000000
mean      2496.778802
std       1774.633152
min         40.000000
25%       1086.775000
50%       2182.800000
75%       3592.475000
max       9553.600000
Name: distance_km, dtype: float64


In [29]:

conditions = [
    df["distance_km"] <= 1000,
    df["distance_km"].between(1000, 2500, inclusive="right"),
    df["distance_km"].between(2500, 4000, inclusive="right"),
    df["distance_km"] > 4000
]

categories = [
    "Short Distance",
    "Medium Distance",
    "Long Distance",
    "Very Long Distance"
]

df["distance_band"] = np.select(
    conditions,
    categories,
    default="Check Data"
)

print(df[["distance_km", "distance_band"]].head(10))


   distance_km       distance_band
0       1579.5     Medium Distance
1       1049.0     Medium Distance
2       4552.6  Very Long Distance
3       7074.9  Very Long Distance
4        522.0      Short Distance
5       3398.8       Long Distance
6        489.6      Short Distance
7       4320.7  Very Long Distance
8       2598.2       Long Distance
9        281.6      Short Distance


In [30]:
print(df["distance_band"].value_counts())

distance_band
Medium Distance       16936
Long Distance         11530
Short Distance        11497
Very Long Distance    10037
Name: count, dtype: int64


In [31]:
print(df["warehouse_processing_hours"].describe())

count    50000.000000
mean        14.412430
std          9.924904
min          1.000000
25%          6.600000
50%         11.900000
75%         20.700000
max         48.000000
Name: warehouse_processing_hours, dtype: float64


In [32]:

conditions = [
    df["warehouse_processing_hours"] <= 6.6,
    df["warehouse_processing_hours"].between(6.6, 11.9, inclusive="right"),
    df["warehouse_processing_hours"].between(11.9, 20.7, inclusive="right"),
    df["warehouse_processing_hours"] > 20.7
]

categories = [
    "Fast",
    "Moderate",
    "Slow",
    "Very Slow"
]

df["warehouse_processing_band"] = np.select(
    conditions,
    categories,
    default="Check Data"
)

print(
    df[["warehouse_processing_hours", "warehouse_processing_band"]].head(10)
)


   warehouse_processing_hours warehouse_processing_band
0                        25.1                 Very Slow
1                        25.9                 Very Slow
2                        31.5                 Very Slow
3                        12.6                      Slow
4                        24.9                 Very Slow
5                        32.2                 Very Slow
6                        21.0                 Very Slow
7                        21.2                 Very Slow
8                        12.7                      Slow
9                        15.1                      Slow


In [34]:
print(df["warehouse_processing_band"].value_counts())

warehouse_processing_band
Fast         12602
Slow         12509
Very Slow    12472
Moderate     12417
Name: count, dtype: int64


In [ ]:
new_features = [
    "order_month",
    "order_day_of_week",
    "delay_severity",
    "shipping_cost_pct_of_order_value",
    "distance_band",
    "warehouse_processing_band"
]

print(df[new_features].head())
print("\nMissing values in new features:")
print(df[new_features].isna().sum())

In [ ]:

output_path = "../data/Supply_Chain_Feature_Engineered.csv"

df.to_csv(output_path, index=False)

print("Dataset saved successfully!")
print("Rows and columns:", df.shape)
print("File saved at:", output_path)
